# Avaliar o modelo fine-tuned
Executa as duas células de código por ordem, com o kernel `.GB`. O CSV mostra cada previsão e os três scores (que somam 1).

**Atenção:** por defeito, estes são os mesmos emails usados no fine-tuning. As métricas medem desempenho no treino, não a capacidade de classificar emails novos. Para uma avaliação independente, aponta `EXCEL` e `EMAIL_DIR` para um conjunto rotulado que não tenha sido usado no treino.


In [ ]:
from pathlib import Path
from openpyxl import load_workbook
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from transformers import pipeline


def find_root():
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / 'research/finetuning.py').exists():
            return folder
    raise FileNotFoundError('Abre o notebook dentro do projeto GlobalBrico.')

ROOT = find_root()
from importlib.util import module_from_spec, spec_from_file_location
spec = spec_from_file_location('finetuning', ROOT / 'research/finetuning.py')
finetuning = module_from_spec(spec)
spec.loader.exec_module(finetuning)
load_examples = finetuning.load_examples
MODEL_DIR = ROOT / 'src/models/xlm_roberta_large_xnli_finetuned'
EXCEL = ROOT / 'research/ground_truth/ground_truth_emails.xlsx'
EMAIL_DIR = ROOT / 'research/extracted_emails'
OUTPUT_CSV = ROOT / 'research/finetuned_evaluation.csv'
LABELS = ['Pedido de Informação', 'Pedido de Encomenda', 'SPAM']


In [ ]:


examples = load_examples(EXCEL, EMAIL_DIR)  # mesmo tratamento de texto usado no treino
sheet = load_workbook(EXCEL, read_only=True, data_only=True)['Revisão']
rows = sheet.iter_rows(values_only=True)
header = next(row for row in rows if 'UID' in row and 'Label correta' in row)
uid_col, label_col = header.index('UID'), header.index('Label correta')
uids = [str(row[uid_col]).removesuffix('.0') for row in rows if row[uid_col] and row[label_col]]
assert len(uids) == len(examples)

classifier = pipeline('text-classification', model=str(MODEL_DIR), tokenizer=str(MODEL_DIR), device=-1)
outputs = classifier([text for text, _ in examples], truncation=True, max_length=256,
                     top_k=None, batch_size=2)
records = []
for uid, (_, correct), scores in zip(uids, examples, outputs):
    by_label = {item['label']: item['score'] for item in scores}
    if set(by_label) != set(LABELS):
        raise ValueError(f'Labels inesperadas no modelo: {list(by_label)}')
    predicted = max(LABELS, key=by_label.get)
    records.append({'uid': uid, 'label_correta': correct, 'label_prevista': predicted,
                    'score': by_label[predicted], **{f'score_{label}': by_label[label] for label in LABELS}})

results = pd.DataFrame(records)
assert len(results) == len(examples)
results.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(classification_report(results.label_correta, results.label_prevista, labels=LABELS, zero_division=0))
print('Matriz de confusão (linhas: correta; colunas: prevista):')
display(pd.DataFrame(confusion_matrix(results.label_correta, results.label_prevista, labels=LABELS),
                     index=LABELS, columns=LABELS))
print('Erros:')
display(results[results.label_correta != results.label_prevista])
print(f'Resultados por email: {OUTPUT_CSV}')


In [ ]:
# Mesmo conjunto de UIDs: XLM-RoBERTa-large-XNLI zero-shot vs. modelo fine-tuned.
from sklearn.metrics import accuracy_score, f1_score, recall_score

BEFORE_CSV = ROOT / 'research/benchmark_results/results/evaluated_predictions.csv'
previous = pd.read_csv(BEFORE_CSV, dtype={'uid': str})
previous = previous[(previous['model'] == 'XLM-RoBERTa-large-XNLI') &
                    (previous['method'] == 'zero_shot_nli')][['uid', 'predicted']]
current = results[['uid', 'label_correta', 'label_prevista']].copy()
current['uid'] = current['uid'].astype(str)
if previous['uid'].duplicated().any() or current['uid'].duplicated().any():
    raise ValueError('UIDs repetidos nas previsões; a comparação exige um email por UID.')
paired = current.merge(previous.rename(columns={'predicted': 'antes'}), on='uid', validate='one_to_one')
if len(paired) != len(current) or len(paired) != len(previous):
    raise ValueError('Os resultados antes/depois não abrangem exatamente os mesmos UIDs.')

def measures(prediction):
    truth = paired['label_correta']
    request = truth != 'SPAM'
    spam = truth == 'SPAM'
    return {
        'Macro-F1': f1_score(truth, prediction, labels=LABELS, average='macro', zero_division=0),
        'Accuracy': accuracy_score(truth, prediction),
        'Pedidos marcados SPAM': int((request & (prediction == 'SPAM')).sum()),
        'SPAM marcado como pedido': int((spam & (prediction != 'SPAM')).sum()),
        **{f'Recall {label}': recall_score(truth, prediction, labels=[label], average=None, zero_division=0)[0]
           for label in LABELS},
    }

before = pd.Series(measures(paired['antes']), name='Antes: XNLI zero-shot')
after = pd.Series(measures(paired['label_prevista']), name='Depois: fine-tuned')
comparison = pd.concat([before, after], axis=1)
comparison['Variação (depois - antes)'] = comparison.iloc[:, 1] - comparison.iloc[:, 0]
display(comparison)
print('Casos que mudaram:')
display(paired.loc[paired['antes'] != paired['label_prevista']])
print('Os resultados depois usam emails do treino; as métricas não demonstram melhoria em emails novos.')
